# 基于MindSpore的GLM-4-9B模型的聊天应用实现

本案例基于MindNLP和GLM-4-9B实现一个聊天应用。**支持流式回复**。

## 环境准备

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [ ]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

## 1. 依赖安装

In [1]:
!pip install gradio mdtex2html -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install ipywidgets -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 6.9 MB/s  0:00:03m0:00:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29/29 [gradio]28/29 [gradio]]-it-py]
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 3.9 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]


In [3]:
# 如果连接失败，请本地下载后上传到华为云
# !wget https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_arm64
!mv frpc_linux_arm64 frpc_linux_arm64_v0.3
!mkdir -p /home/ma-user/.cache/huggingface/gradio/frpc
!mv frpc_linux_arm64_v0.3 /home/ma-user/.cache/huggingface/gradio/frpc/
!chmod +x /home/ma-user/.cache/huggingface/gradio/frpc/frpc_linux_arm64_v0.3

In [8]:
# 降低diffusers版本
!pip install diffusers==0.35.2

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 89.1 MB/s  0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.36.0
    Uninstalling diffusers-0.36.0:
      Successfully uninstalled diffusers-0.36.0


## 2. 代码开发

In [1]:
import mindspore
import mindnlp
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    StoppingCriteria,
    StoppingCriteriaList,
    BitsAndBytesConfig
)
import gradio as gr
import mdtex2html
import mindspore

mindspore.launch_blocking()
mindspore.set_context(device_target='Ascend')

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
Modular Diffusers is currently an experimental feature under active development. The API is su

In [2]:
# 加载tokenizer和模型
tokenizer = AutoTokenizer.from_pretrained('THUDM/glm-4-9b-chat-hf')
model = AutoModelForCausalLM.from_pretrained('THUDM/glm-4-9b-chat-hf')

# 确保tokenizer有pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
class StopOnTokens(StoppingCriteria):
    """自定义停止条件"""
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        stop_ids = self.model.config.eos_token_id
        if not isinstance(stop_ids, (list, tuple)):
            stop_ids = [stop_ids]
        for stop_id in stop_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

def glm4_stream_chat(model, tokenizer, query, history=None, max_length=8192, 
                    top_p=0.8, temperature=0.6, timeout=300):
    """GLM-4-9B的流式对话方法（无线程版本）"""
    
    if history is None:
        history = []
    
    # 构建消息历史
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": query})
    
    # 应用聊天模板并获取attention_mask
    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="ms",
        return_dict=True
    )
    
    input_ids = model_inputs["input_ids"]
    attention_mask = model_inputs.get("attention_mask", None)
    
    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)
    
    # 设置生成参数
    stop_criteria = StopOnTokens(model)
    
    generate_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": max_length,
        "do_sample": True,
        "top_p": top_p,
        "temperature": temperature,
        "stopping_criteria": StoppingCriteriaList([stop_criteria]),
        "repetition_penalty": 1.2,
        "eos_token_id": model.config.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }
    
    # 直接生成而不使用线程和流式处理器
    # 使用generate方法生成完整的响应
    generated_ids = model.generate(**generate_kwargs)
    
    # 提取新生成的部分（排除输入部分）
    new_tokens = generated_ids[0][input_ids.shape[1]:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # 模拟流式输出：将生成的文本分成小块逐步返回
    # 这样可以保持流式体验但不需要线程
    chunk_size = 3  # 每次返回的字符数
    current_text = ""
    
    for i in range(0, len(generated_text), chunk_size):
        chunk = generated_text[i:i + chunk_size]
        current_text += chunk
        
        # 添加小延迟以模拟流式效果
        import time
        time.sleep(0.01)
        
        yield current_text, history + [(query, current_text)]
    
    # 更新历史记录
    history.append((query, generated_text))

In [4]:
import os
from typing import Dict, Union, Optional
from typing import List


import mdtex2html

#将文本中的字符转为网页上可以支持的字符，避免被误认为是HTML标签
def parse_text(text):
    """copy from https://github.com/GaiZhenbiao/ChuanhuChatGPT/"""
    lines = text.split("\n")
    lines = [line for line in lines if line != ""]
    count = 0
    for i, line in enumerate(lines):
        if "```" in line:
            count += 1
            items = line.split('`')
            if count % 2 == 1:
                lines[i] = f'<pre><code class="language-{items[-1]}">'
            else:
                lines[i] = f'<br></code></pre>'
        else:
            if i > 0:
                if count % 2 == 1:
                    line = line.replace("`", "\`")
                    line = line.replace("<", "&lt;")
                    line = line.replace(">", "&gt;")
                    line = line.replace(" ", "&nbsp;")
                    line = line.replace("*", "&ast;")
                    line = line.replace("_", "&lowbar;")
                    line = line.replace("-", "&#45;")
                    line = line.replace(".", "&#46;")
                    line = line.replace("!", "&#33;")
                    line = line.replace("(", "&#40;")
                    line = line.replace(")", "&#41;")
                    line = line.replace("$", "&#36;")
                lines[i] = "<br>"+line
    text = "".join(lines)
    return text

## 3 基于 Gradio 创建聊天应用

In [5]:
# 编写Gradio调用函数

#采用流聊天方式（stream_chat）调用模型，使得生成答案有逐字生成的效果
def predict(input, chatbot, max_length, top_p, temperature, history):
    chatbot.append((parse_text(input), ""))
    for full_response, current_history in glm4_stream_chat(model, tokenizer, input, history):
        # 只打印新增的内容
        new_content = full_response[current_length:]
        print(new_content, end="", flush=True)
        current_length = len(full_response)
        chatbot[-1] = (parse_text(input), parse_text(full_response))       
        history = current_history  # 更新历史记录
        yield chatbot, history

#去除输入框的内容
def reset_user_input():
    return gr.update(value='')

#清除状态
def reset_state():
    return [], [], None

In [6]:
#运行Gradio界面，运行成功后点击“Running on public URL”后的网页链接即可体验
import gradio as gr

with gr.Blocks() as demo:
    gr.HTML("""<h1 align="center">MindNLP GLM4-9b StreamChat</h1>""")

    chatbot = gr.Chatbot()
    with gr.Row():
        with gr.Column(scale=4):
            with gr.Column(scale=12):
                user_input = gr.Textbox(show_label=False, placeholder="Input...", lines=3, container=False)
            with gr.Column(min_width=32, scale=1):
                with gr.Row():
                    submitBtn = gr.Button("一键开聊", variant="primary")
                    emptyBtn = gr.Button("清除历史")
            with gr.Column(scale=1):
                max_length = gr.Slider(0, 4096, value=2048, step=1.0, label="Maximum length", interactive=True)
                top_p = gr.Slider(0, 1, value=0.7, step=0.01, label="Top P", interactive=True)
                temperature = gr.Slider(0, 1, value=0.95, step=0.01, label="Temperature", interactive=True)               

    history = gr.State([])

    submitBtn.click(predict, [user_input, chatbot, max_length, top_p, temperature, history], [chatbot, history],
                    show_progress=True)
    submitBtn.click(reset_user_input, [], [user_input])

    emptyBtn.click(reset_state, outputs=[chatbot, history], show_progress=True)

demo.queue().launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8a47de923e2017aa58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
